### Generating ML-safe field names and rename mapping

In [ ]:
# Purpose: Generating ML-safe field names and rename mapping dictionary.

import re

original_fields = [
    "Unnamed: 0", "GlobalID", "Census2020_INTPTLAT20", "Census2020_INTPTLON20",
    "California_FireName", "CalEnviroS_ZIP", "CalEnviroS_Population", "CalEnviroS_DieselPm",
    "CalEnviroS_Traffic", "CalEnviroS_ImpWaterBodies", "CalEnviroS_Education",
    "CALFIREFor_ForestDist", "California_COUNTYFP", "California_CLASSFP",
    "California_ALAND", "California_AWATER", "California_OwnLevel", "FireHazard_Fhsz7class",
    "Total ETo (in)", "Total Precip (in)", "Avg Vap Pres (mBars)", "Avg Max Air Temp (F)",
    "Avg Min Air Temp (F)", "Avg Air Temp (F)", "Avg Max Rel Hum (%)", "Avg Min Rel Hum (%)",
    "Avg Rel Hum (%)", "Avg Dew Point (F)", "Avg Wind Speed (mph)", "Avg Soil Temp (F)",
    "Site ID", "Yr Avg PM2.5 Concentration", "Yr Avg AQI Value", "Date", "FireBinary", "GIS_FireAcres"
]

def make_safe(name):
    # Lowercasing, replacing non-alphanumeric with underscore, and collapsing repeats
    s = name.lower()
    s = re.sub(r"[^0-9a-z]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    # Ensuring name does not start with a digit
    if re.match(r"^[0-9]", s):
        s = f"f_{s}"
    return s

safe_fields = [make_safe(f) for f in original_fields]

rename_map = dict(zip(original_fields, safe_fields))

print("Safe field names mapping:")
for k, v in rename_map.items():
    print(f"{k:40s} -> {v}")


### Unified run directory and structured logging

In [ ]:
# Purpose: Unified run directory and structured logging.

import os
from pathlib import Path
from datetime import datetime

BASE_DIR = Path.cwd()
RUNS_DIR = BASE_DIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)

# Creating a timestamped run directory
TS = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = Path(
    os.environ.get("ML_RUN_DIR", RUNS_DIR / f"run-{TS}")
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Using run directory:", RUN_DIR)


### Load dataset, handle missing values, and scaling

In [ ]:
# Purpose: Load dataset safely, handle missing values, and apply scaling.

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import zscore

# Adjusting this path to point to your prepared CSV
DATA_PATH = Path("CalWildFire_ML_Joined_Data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

print("Loading dataset from:", DATA_PATH)
df_raw = pd.read_csv(DATA_PATH)

print("Original shape:", df_raw.shape)

# Renaming columns using mapping from earlier cell (rename_map)
try:
    df = df_raw.rename(columns=rename_map)
except NameError:
    # Fallback if rename_map is not in scope
    df = df_raw.copy()
    print("Warning: rename_map not found, keeping original column names.")

# Checking for FireBinary and dropping rows with missing target
if "FireBinary" not in df.columns:
    raise KeyError("Expected 'FireBinary' column not found in dataset.")

df = df.dropna(subset=["FireBinary"])
y = df["FireBinary"].astype(int)

# Dropping non-feature columns if present
drop_cols = [c for c in ["firebinary", "FireBinary", "globalid", "Unnamed: 0"] if c in df.columns]
X = df.drop(columns=drop_cols, errors="ignore").copy()

print("Shape after dropping target/id columns:", X.shape)

# Handling missing values: using median for numeric and most_frequent for categorical
num_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.number)]
cat_cols = [c for c in X.columns if c not in num_cols]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

if num_cols:
    X[num_cols] = num_imputer.fit_transform(X[num_cols])

if cat_cols:
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# Optional: removing extreme outliers using z-score on numeric fields
if num_cols:
    z = np.abs(zscore(X[num_cols], nan_policy="omit"))
    # Using a simple threshold; can be adjusted
    mask = (z < 5).all(axis=1)
    print("Keeping rows after outlier filter:", mask.sum(), "out of", len(mask))
    X = X.loc[mask].reset_index(drop=True)
    y = y.loc[mask].reset_index(drop=True)

print("Final cleaned shape:", X.shape)

# Applying MinMax scaling on numeric columns only (keeping categorical raw for encoders later)
scaler = MinMaxScaler()
if num_cols:
    X[num_cols] = scaler.fit_transform(X[num_cols])

print("Finished cleaning and scaling feature matrix.")

gc.collect()


### Train RF and Logistic Regression (saga) with CV and hold-out

In [ ]:
# Purpose: Train RF and LogReg(saga) with CV and hold-out, save all predictions/metrics/models.
# Fix: ColumnTransformer for LogReg now correctly specifies categorical columns (cat_cols_lin),
# preventing "expected 3, got 2" error.

import gc, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import joblib

# -----------------------------
# Output dir
# -----------------------------
try:
    OUT_DIR = RUN_DIR
except NameError:
    OUT_DIR = Path("runs") / f"manual_run-{datetime.now():%Y%m%d-%H%M%S}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_df(df, name):
    p = OUT_DIR / f"{name}.csv"
    df.to_csv(p, index=False)
    print("Saved:", p)

def save_json(obj, name):
    p = OUT_DIR / f"{name}.json"
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    print("Saved:", p)

def save_pkl(obj, name):
    joblib.dump(obj, OUT_DIR / f"{name}.pkl")
    print(f"Saved model: {name}")

# -----------------------------
# Feature split
# -----------------------------
# Drop ID-like columns if present; y should already be defined in previous cells.
drop_cols = [c for c in ["Unnamed: 0", "GlobalID"] if c in X.columns]
X_use = X.drop(columns=drop_cols, errors="ignore").copy()
feature_cols = [c for c in X_use.columns if c != "FireBinary"]

# Determine categorical vs numeric based on dtype
cat_cols_rf = [c for c in feature_cols if str(X_use[c].dtype) in ("category", "object", "string")]
num_cols    = [c for c in feature_cols if c not in cat_cols_rf]

# For linear model we use the same categorical list (OHE will handle them)
cat_cols_lin = cat_cols_rf[:]

def make_ohe_sparse():
    """
    Creating OHE that works across sklearn versions.
    Trying to use sparse output and min_frequency where available.
    """
    try:
        # sklearn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=0.01)
    except TypeError:
        try:
            # sklearn <= 1.1 with min_frequency
            return OneHotEncoder(handle_unknown="ignore", sparse=True, min_frequency=0.01)
        except TypeError:
            # very old sklearn without min_frequency
            return OneHotEncoder(handle_unknown="ignore", sparse=True)

ohe_sparse = make_ohe_sparse()

# -----------------------------
# Pipelines
# -----------------------------

# RandomForest: OrdinalEncoder for categoricals, passthrough numerics
rf_pre = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols_rf),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

rf_model = Pipeline(steps=[
    ("prep", rf_pre),
    ("rf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=123, max_features="sqrt"))
])

# Logistic Regression: Sparse OHE on categoricals + StandardScaler on numerics (sparse-friendly)
num_pipe_lin = Pipeline([("scale", StandardScaler(with_mean=False))])

lin_pre = ColumnTransformer(
    transformers=[
        ("cat", ohe_sparse, cat_cols_lin),
        ("num", num_pipe_lin, num_cols),
    ],
    remainder="drop"
)

lin_model = Pipeline(steps=[
    ("prep", lin_pre),
    ("logreg", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=0.5,
        tol=1e-3,
        max_iter=2000,
        class_weight="balanced"
    ))
])

# -----------------------------
# CV utilities
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

def _get_scores(estimator, Xmat):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(Xmat)[:, 1]
    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(Xmat)
        return s if s.ndim == 1 else s[:, 1]
    return estimator.predict(Xmat)

def _metrics(y_true, y_pred, y_score):
    return dict(
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision=float(precision_score(y_true, y_pred, zero_division=0)),
        recall=float(recall_score(y_true, y_pred, zero_division=0)),
        f1=float(f1_score(y_true, y_pred, zero_division=0)),
        roc_auc=float(roc_auc_score(y_true, y_score))
    )

def cv_eval(model, Xdf, y, label):
    all_scores = []
    preds_list = []
    for f_idx, (tr, va) in enumerate(cv.split(Xdf, y), start=1):
        print(f"[{label}] Fold {f_idx}")
        Xtr, Xva = Xdf.iloc[tr], Xdf.iloc[va]
        ytr, yva = y.iloc[tr], y.iloc[va]

        model.fit(Xtr, ytr)
        ypred = model.predict(Xva)
        yscore = _get_scores(model, Xva)

        all_scores.append(_metrics(yva, ypred, yscore))
        preds_list.append(pd.DataFrame({
            "y_true": yva.to_numpy(),
            "y_pred": ypred.astype(int),
            "y_score": yscore.astype(float),
            "fold": f_idx
        }))

        # save fold model
        save_pkl(model, f"{label.lower()}_fold_{f_idx}")

        gc.collect()

    preds_df = pd.concat(preds_list, axis=0, ignore_index=True)
    mean = {k: float(np.mean([sc[k] for sc in all_scores])) for k in all_scores[0]}
    save_df(preds_df, f"cv_predictions_{label.lower()}")
    save_json(mean, f"cv_metrics_{label.lower()}")
    return mean

# -----------------------------
# Run CV for both models
# -----------------------------
rf_cv = cv_eval(rf_model, X_use, y, "RandomForest")
lin_cv = cv_eval(lin_model, X_use, y, "LogReg_saga")

# -----------------------------
# Hold-out evaluation
# -----------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X_use[feature_cols], y, test_size=0.2, stratify=y, random_state=123
)

# RandomForest hold-out
rf_model.fit(X_tr, y_tr)
y_pred_rf = rf_model.predict(X_te)
y_score_rf = _get_scores(rf_model, X_te)
save_df(pd.DataFrame({"y_true": y_te.to_numpy(),
                      "y_pred": y_pred_rf.astype(int),
                      "y_score": y_score_rf.astype(float)}),
        "holdout_predictions_rf")
save_json(_metrics(y_te, y_pred_rf, y_score_rf), "holdout_metrics_rf")
save_pkl(rf_model, "rf_model_final")

# Logistic Regression hold-out
lin_model.fit(X_tr, y_tr)
y_pred_lin = lin_model.predict(X_te)
y_score_lin = _get_scores(lin_model, X_te)
save_df(pd.DataFrame({"y_true": y_te.to_numpy(),
                      "y_pred": y_pred_lin.astype(int),
                      "y_score": y_score_lin.astype(float)}),
        "holdout_predictions_logreg_saga")
save_json(_metrics(y_te, y_pred_lin, y_score_lin), "holdout_metrics_logreg_saga")
save_pkl(lin_model, "lin_model_final")

print("All outputs saved to:", OUT_DIR)
